In [1]:
import pandas as pd
import numpy as np
import pyarrow

In [ ]:
df = pd.read_parquet("merged_data.parquet").copy()

In [3]:
df.columns

Index(['Protocollo', 'Gruppo', 'DataOraIncidente', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'NaturaIncidente', 'particolaritastrade', 'TipoStrada',
       'FondoStradale', 'Pavimentazione', 'Segnaletica',
       'CondizioneAtmosferica', 'Traffico', 'Visibilita', 'Illuminazione',
       'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI', 'Longitude',
       'Latitude', 'Confermato', 'Progressivo', 'TipoVeicolo', 'StatoVeicolo',
       'TipoPersona', 'Sesso', 'Tipolesione', 'Deceduto', 'DecedutoDopo',
       'CinturaCascoUtilizzato', 'Airbag'],
      dtype='object')

In [4]:
df.shape

(685881, 37)

In [5]:
columns_categories = ['Localizzazione1',
                      'NaturaIncidente',
                      'particolaritastrade',
                      'TipoStrada',
                      'FondoStradale',
                      'Pavimentazione',
                      'Segnaletica',
                      'CondizioneAtmosferica',
                      'Traffico',
                      'Visibilita',
                      'Illuminazione',
                      'TipoVeicolo',
                      'StatoVeicolo',
                      'TipoPersona',
                      'Sesso',
                      'Tipolesione',
                      'DecedutoDopo',
                      'CinturaCascoUtilizzato',
                      'Airbag']

columns_int64 = ['Gruppo',
                 'NUM_FERITI',
                 'NUM_RISERVATA',
                 'NUM_MORTI',
                 'NUM_ILLESI',
                 'Confermato',
                 'Progressivo',
                 'Deceduto']

options_sets = {}
options_lists = {}

all_columns_limited_options = columns_int64 + columns_categories

for col in all_columns_limited_options:
    options_sets[f'{col}_options'] = set(df[col].dropna().unique())
    options_lists[f'{col}_list'] = sorted(options_sets[f"{col}_options"])
    print(f"{col}_list:", options_lists[f"{col}_list"])

Gruppo_list: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(23), np.int64(26)]
NUM_FERITI_list: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(17), np.int64(18), np.int64(25), np.int64(65)]
NUM_RISERVATA_list: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
NUM_MORTI_list: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]
NUM_ILLESI_list: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(17), np.int64(18)]
Confermato_list: [np.int64(-1), np.int64(0)]
Progress

This allows us to double-check that there are no more issues with misaligned rows. Each row is now aligned with the correct column header.

DATA CLEANING: DUPLICATE ROWS

In [6]:
duplicate_rows = df[df.duplicated()]
duplicate_rows.shape

(20093, 37)

There are 20093 out of 685881 rows (2.9%) with duplicate data! Let's take a closer look:

In [7]:
duplicate_rows.head()

,Protocollo,Gruppo,DataOraIncidente,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare,...,Progressivo,TipoVeicolo,StatoVeicolo,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag
2,1931569,7,01/01/2013 13:15,Strada Urbana,VIA DEI CICLAMINI,all'intersezione con,VIA DELLE ROSE,VIA DELLE ROSE,,,...,1,Autovettura privata,In marcia / fermata / arresto,Passeggero,M,Illeso,0,,Non accertato,Inesistente
14,1931576,11,01/01/2013 13:00,Strada Urbana,VIA OSTIENSE,in prossimitÃ,,del civico n.,50,,...,2,Autovettura privata,In marcia / fermata / arresto,Passeggero,M,Illeso,0,,Utilizzato,Inesistente
31,1931590,11,01/01/2013 01:45,Strada Urbana,VIA DEL PORTO FLUVIALE,in corrispondenza,,del civico n.,3C,,...,1,Autovettura privata,In marcia / fermata / arresto,Passeggero,F,Illeso,0,,Utilizzato,Inesistente
72,1931629,5,01/01/2013 03:00,Strada Urbana,VIA DEI MONTI TIBURTINI,all'intersezione semaforizzata con,VIA DEL TUFO,VIA DEL TUFO,,,...,3,Autovettura privata,In marcia / fermata / arresto,Passeggero,F,Illeso,0,,Non accertato,Inesistente
107,1931646,13,02/01/2013 12:50,Strada Urbana,VIA ARISTIDE CARABELLI,all'intersezione con,VIA CORRADO DEL GRECO,VIA CORRADO DEL GRECO,,,...,2,Autovettura privata,In marcia / fermata / arresto,Passeggero,M,Illeso,0,,Non accertato,Inesistente


The first 5 are passengers ('TipoPersona' is 'Passeggero'). Let's see if all of these are. These passengers are likely to be valid duplicates;  it means there were several (identical) passengers in the vehicle. Filtering out passengers:

In [8]:
duplicate_rows_not_passenger = duplicate_rows[duplicate_rows['TipoPersona'] != 'Passeggero']
duplicate_rows_not_passenger

,Protocollo,Gruppo,DataOraIncidente,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare,...,Progressivo,TipoVeicolo,StatoVeicolo,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag
136,1931673,1,01/01/2013 02:15,Strada Urbana,VIA EMILIA,in corrispondenza,,del civico n.,104,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,
760,1931947,10,06/01/2013 18:40,Strada Urbana,VIA CALPURNIO FIAMMA,in prossimitÃ,,del civico n.,19,,...,<NA>,,,Pedone,F,Rifiuta cure immediate,0,NON DECEDUTO,,
761,1931947,10,06/01/2013 18:40,Strada Urbana,VIA CALPURNIO FIAMMA,in prossimitÃ,,del civico n.,19,,...,<NA>,,,Pedone,F,Rifiuta cure immediate,0,NON DECEDUTO,,
797,1931963,2,05/01/2013 20:00,Strada Urbana,VIA TEMBIEN,in prossimitÃ,,del civico n.,33,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,
3083,1933027,15,17/01/2013 19:00,Strada Urbana,VIA PORTUENSE,all'intersezione semaforizzata con,VIA USINI,VIA USINI,,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683217,6075416,19,04/08/2022 09:30:00,Strada Urbana,VIA LORENZO ELLERO,in prossimitÃ,,del civico n.,29,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,
683839,6080058,11,08/08/2022 10:50:00,Strada Urbana,VIALE PICO DELLA MIRANDOLA,all'intersezione semaforizzata con,PIAZZALE DEI CADUTI DELLA MONTAGNOLA,PIAZZALE DEI CADUTI DELLA MONTAGNOLA,,,...,<NA>,,,Pedone,F,Rimandato,0,NON DECEDUTO,,
683871,6080163,1,07/08/2022 21:45:00,Strada Urbana,VIA GIOBERTI,all'intersezione con,VIA FILIPPO TURATI,VIA FILIPPO TURATI,,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,
685303,6094774,20,25/08/2022 18:30:00,Strada Urbana,VIA DI GROTTAROSSA,all'intersezione con,VIA DI QUARTO ANNUNZIATA,VIA DI QUARTO ANNUNZIATA,,,...,<NA>,,,Pedone,M,Rimandato,0,NON DECEDUTO,,


We are left with 578 rows. The ones visible here are all pedestrians ('TipoPersona' is 'Pedone').

In [9]:
duplicate_rows_not_passenger_not_pedestrian = duplicate_rows_not_passenger[
    duplicate_rows_not_passenger['TipoPersona'] != 'Pedone']
duplicate_rows_not_passenger_not_pedestrian

,Protocollo,Gruppo,DataOraIncidente,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare,...,Progressivo,TipoVeicolo,StatoVeicolo,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag
10007,1936050,20,15/02/2013 23:55,Strada Urbana,GALLERIA GIOVANNI XXIII,da specificare,,,,Rampa di accesso da via Acquedotto del Peschiera,...,1,Autovettura privata,In marcia / fermata / arresto,Passeggero Istruttore,M,Illeso,0,,Non accertato,Inesistente
179524,2397352,16,07/06/2015 23:30,Strada Urbana,VIALE GAETANO ARTURO CROCCO,in prossimitÃ,,del civico n.,FC 71,,...,3,Autovettura privata,Allontanatosi,Conducente,M,Illeso,0,,,


Now we have two rows left to check for duplicates: 
- Row 10007 again is a passenger, but was not filtered out previously as he is a 'passenger instructor'; this is a valid duplicate. 
- Row 179524 is a driver with 'progressivo' 3. If we closely examine this protocol, number 2397352, we can understand if this is a valid duplicate or not. 

In [10]:
df[df['Protocollo'] == 2397352]

,Protocollo,Gruppo,DataOraIncidente,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare,...,Progressivo,TipoVeicolo,StatoVeicolo,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag
179521,2397352,16,07/06/2015 23:30,Strada Urbana,VIALE GAETANO ARTURO CROCCO,in prossimitÃ,,del civico n.,FC 71,,...,1,Autovettura privata,In marcia / fermata / arresto,Conducente,M,Illeso,0,,,
179522,2397352,16,07/06/2015 23:30,Strada Urbana,VIALE GAETANO ARTURO CROCCO,in prossimitÃ,,del civico n.,FC 71,,...,1,Autovettura privata,In marcia / fermata / arresto,Passeggero,F,Illeso,0,,,
179523,2397352,16,07/06/2015 23:30,Strada Urbana,VIALE GAETANO ARTURO CROCCO,in prossimitÃ,,del civico n.,FC 71,,...,3,Autovettura privata,Allontanatosi,Conducente,M,Illeso,0,,,
179524,2397352,16,07/06/2015 23:30,Strada Urbana,VIALE GAETANO ARTURO CROCCO,in prossimitÃ,,del civico n.,FC 71,,...,3,Autovettura privata,Allontanatosi,Conducente,M,Illeso,0,,,


This accident seems to involve three parties: a male car driver with a female passenger, a male car driver who did not stop and another male car driver who did not stop. There appears to be an incorrect 'Progressivo' number for row 179523, which led to row 179524 showing up as a duplicate. This accident did not involve a pedestrian, so we will remove this accident from our dataframe:

In [11]:
df = df[df['Protocollo'] != 2397352]

In [12]:
df.shape

(685877, 37)

Now the number of rows has gone from 685881 to 685877 (4 less), as expected. Now we reset the index as rows have been deleted:

In [13]:
df = df.reset_index(drop=True)

DATA CLEANING: DUPLICATE ROWS - COMPLETE!

In [14]:
df.to_parquet('data_no_duplicates.parquet', index=False)

In [ ]:
metadata = {
    'notebook': '003 Data cleaning duplicates.ipynb',
    'step': 'duplicate rows cleanup',
    'initial_parquet_file': 'merged_data.parquet',
    'initial_rows': 685_881,
    'initial_columns': 37,
    'duplicate_rows_found': 20_093,
    'duplicate_share_percent': 2.93,  # 20,093 / 685,881
    'duplicate_detection_method': "pandas.DataFrame.duplicated() across all columns",
    'investigation_filters': {
        'excluded_roles_first_pass': ['Passeggero'],  # passengers
        # pedestrians (to isolate anomalies)
        'excluded_roles_second_pass': ['Pedone']
    },
    'records_removed_protocollo': [2397352],
    'rows_removed': 4,
    'final_rows': 685_877,
    'final_columns': 37,
    'index_reset': True,
    'final_parquet_file': 'data_no_duplicates.parquet',
    'columns_added': [],
    'columns_removed': [],
    'decisions_made': (
        "Identified 20,093 duplicate rows (2.9%). "
        "Most duplicates involved passengers; excluded these from duplicate review. "
        "Temporarily excluded pedestrians to surface remaining anomalies. "
        "Manually inspected cases; removed accident Protocollo 2397352 (4 rows). "
        "Reset index and saved cleaned dataset."
    )
}